In [73]:
%load_ext autoreload
%autoreload 2

from models import *
from helpers import *
from torchattacks import PGD, FGSM

import os, sys
current_dir = os.getcwd()
path_to_append = os.path.join(current_dir, "configs")
if path_to_append not in sys.path:
    sys.path.append(path_to_append)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
from importlib import import_module
def load_configs_from_file(config_file):
    configs = import_module(config_file.split(".")[0])
    
    return configs.model_path, configs.hiddens_config, configs.batch_size, configs.epsilon, configs.T, configs.c, configs.lr, configs.lr_sigma, \
        configs.lr_c, configs.perturb_freq, configs.noise_scale, configs.grad_threshold, configs.c_reduce_rate

# config_filename = "adv_MNIST_clean_config" 
config_filename = "adv_MNIST_pgd_config" 

input_dim, input_size = 1, 28 # MNIST images
model_path, hiddens_config, batch_size, epsilon, T, c, lr, lr_sigma, lr_c, perturb_freq, noise_scale, grad_threshold, c_reduce_rate  = load_configs_from_file(config_filename)


In [13]:
model = CNN(input_size, input_dim,
            hiddens_config)
model.load_state_dict(torch.load(model_path))
print ("Classifier loaded!\nEvaluating...")

_, test_loader, _ = load_MNIST_dataset(batch_size = batch_size)

entropy_loss = nn.CrossEntropyLoss()
fc_model = convert_CNN_to_FC(model, [28, 28, 1])

"""Initializing PGD attacks"""
pgd_atk = PGD(model, eps=epsilon, steps=1000)

num_correct, num_all = 0, 0
for images, labels in test_loader: 
    pred_logits = model(images.cuda())
    num_all += labels.shape[0]
    num_correct += (torch.argmax(pred_logits, axis = 1) == labels.cuda()).sum()
print ("\nValidation accuracy: {:.3f}%".format(num_correct / num_all * 100))


Classifier loaded!
Evaluating...

Validation accuracy: 98.940%


In [30]:
model = CNN(input_size, input_dim, hiddens_config)
model.load_state_dict(torch.load(model_path))
print("Classifier loaded!\nEvaluating...")

_, test_loader, _ = load_MNIST_dataset(batch_size=batch_size)

entropy_loss = nn.CrossEntropyLoss()
fc_model = convert_CNN_to_FC(model, [28, 28, 1])

# PGD attack
pgd_atk = PGD(model, eps=epsilon, steps=1000)

attack_results = []
success_count = 0

for i in range(10):
    
    image, label = draw_image_randomly(test_loader)
    
    clean_logits = model(image.reshape([1, *image.shape]))
    clean_loss = entropy_loss(clean_logits, label.reshape([1, *label.shape]))
    clean_label = torch.argmax(clean_logits[0])
    
    attacked_image = pgd_atk(image.reshape([1, *image.shape]), label.reshape([1, *label.shape]))
    pgd_logits = model(attacked_image)
    pgd_loss = entropy_loss(pgd_logits[0], label)
    pgd_label = torch.argmax(pgd_logits[0])
    
    success = int(pgd_label != label)
    success_count += success
    
    attack_results.append({
        "index": i + 1,
        "clean_label": clean_label.item(),
        "true_label": label.item(),
        "pgd_label": pgd_label.item(),
        "clean_loss": clean_loss.item(),
        "pgd_loss": pgd_loss.item(),
        "success": success
    })

    print(f"Attack {i + 1}:")
    print(f"\tClean classification label: {clean_label}")
    print(f"\tGround truth label: {label}")
    print(f"\tPGD classification label: {pgd_label}")
    print(f"\tClean objective: {clean_loss:.3f}")
    print(f"\tPGD objective: {pgd_loss:.3f}")
    print(f"\tAttack Success: {'Yes' if success else 'No'}\n")

success_rate = (success_count / 10) * 100
print(f"Total Attack Success Rate: {success_rate:.2f}%")


Classifier loaded!
Evaluating...
Attack 1:
	Clean classification label: 2
	Ground truth label: 2
	PGD classification label: 2
	Clean objective: 0.000
	PGD objective: 0.002
	Attack Success: No

Attack 2:
	Clean classification label: 8
	Ground truth label: 8
	PGD classification label: 8
	Clean objective: 0.000
	PGD objective: 0.027
	Attack Success: No

Attack 3:
	Clean classification label: 2
	Ground truth label: 2
	PGD classification label: 8
	Clean objective: 0.000
	PGD objective: 0.810
	Attack Success: Yes

Attack 4:
	Clean classification label: 1
	Ground truth label: 1
	PGD classification label: 1
	Clean objective: 0.000
	PGD objective: 0.034
	Attack Success: No

Attack 5:
	Clean classification label: 9
	Ground truth label: 9
	PGD classification label: 4
	Clean objective: 0.002
	PGD objective: 1.094
	Attack Success: Yes

Attack 6:
	Clean classification label: 6
	Ground truth label: 6
	PGD classification label: 6
	Clean objective: 0.001
	PGD objective: 0.076
	Attack Success: No

Attac

In [31]:
model = CNN(input_size, input_dim, hiddens_config)
model.load_state_dict(torch.load(model_path))
print("Classifier loaded!\nEvaluating...")

_, test_loader, _ = load_MNIST_dataset(batch_size=batch_size)

entropy_loss = nn.CrossEntropyLoss()
fc_model = convert_CNN_to_FC(model, [28, 28, 1])

attack_results = []
success_count = 0

epsilon = 0.2
T = 500
lr = 10e-1
lr_c = 1e-4
lr_sigma = 2.5e-1
c = 0.5
perturb_freq = 5
noise_scale = 1e-3
grad_threshold = 2
c_reduce_rate = 1e-3

for i in range(10):
    image, label = draw_image_randomly(test_loader)
    
    clean_logits = model(image.reshape([1, *image.shape]))
    clean_loss = entropy_loss(clean_logits, label.reshape([1, *label.shape]))
    clean_label = torch.argmax(clean_logits[0])
    
    attacked_image_ncvx, obj_cvx = adversarial_attack_nonconvexOpt(
        image.reshape([-1]).cuda(),
        label.cuda(),
        fc_model,
        epsilon,
        T,
        lr,
        lr_sigma,
        c=c,
        lr_c=lr_c,
        perturb_freq=perturb_freq,
        noise_scale=noise_scale,
        grad_threshold=grad_threshold,
        c_reduce_rate=c_reduce_rate,
        set_proper_sigma_freq=16,
    )
    
    adagd_logits = model(attacked_image_ncvx.reshape([1, 28, 28]))
    adagd_label = torch.argmax(adagd_logits[0])
    adagd_loss = entropy_loss(adagd_logits[0], label)
    
    success = int(adagd_label != label)
    success_count += success
    
    attack_results.append({
        "index": i + 1,
        "clean_label": clean_label.item(),
        "true_label": label.item(),
        "adagd_label": adagd_label.item(),
        "clean_loss": clean_loss.item(),
        "adagd_loss": adagd_loss.item(),
        "success": success
    })
    
    print(f"Attack {i + 1}:")
    print(f"\tClean classification label: {clean_label}")
    print(f"\tGround truth label: {label}")
    print(f"\tADR-GD classification label: {adagd_label}")
    print(f"\tClean objective: {clean_loss:.3f}")
    print(f"\tADR-GD objective: {adagd_loss:.3f}")
    print(f"\tAttack Success: {'Yes' if success else 'No'}\n")

success_rate = (success_count / 10) * 100
print(f"Total Attack Success Rate: {success_rate:.2f}%")


Classifier loaded!
Evaluating...
Attack 1:
	Clean classification label: 0
	Ground truth label: 0
	ADR-GD classification label: 0
	Clean objective: 0.000
	ADR-GD objective: 0.028
	Attack Success: No

Attack 2:
	Clean classification label: 5
	Ground truth label: 5
	ADR-GD classification label: 3
	Clean objective: 0.204
	ADR-GD objective: 4.138
	Attack Success: Yes

Attack 3:
	Clean classification label: 7
	Ground truth label: 7
	ADR-GD classification label: 4
	Clean objective: 0.000
	ADR-GD objective: 1.465
	Attack Success: Yes

Attack 4:
	Clean classification label: 8
	Ground truth label: 8
	ADR-GD classification label: 3
	Clean objective: 0.000
	ADR-GD objective: 3.882
	Attack Success: Yes

Attack 5:
	Clean classification label: 3
	Ground truth label: 3
	ADR-GD classification label: 5
	Clean objective: 0.000
	ADR-GD objective: 4.408
	Attack Success: Yes

Attack 6:
	Clean classification label: 2
	Ground truth label: 2
	ADR-GD classification label: 5
	Clean objective: 0.000
	ADR-GD objec

In [32]:
model = CNN(input_size, input_dim, hiddens_config)
model.load_state_dict(torch.load(model_path))
print("Classifier loaded!\nEvaluating...")

_, test_loader, _ = load_MNIST_dataset(batch_size=batch_size)

entropy_loss = nn.CrossEntropyLoss()
fc_model = convert_CNN_to_FC(model, [28, 28, 1])

attack_results = []
success_count = 0

#  UODR-GD attack
epsilon = 0.25
T = 1000
lr = 10e-1
lr_c = 1e-4
lr_sigma = 2.5e-1
c = 0.5
perturb_freq = 5
noise_scale = 1e-3
grad_threshold = 2
c_reduce_rate = 1e-3

for i in range(10):

    image, label = draw_image_randomly(test_loader)
    
    clean_logits = model(image.reshape([1, *image.shape]))
    clean_loss = entropy_loss(clean_logits, label.reshape([1, *label.shape]))
    clean_label = torch.argmax(clean_logits[0])
    
    attacked_image_ncvx, obj_cvx = adversarial_attack_nonconvexOpt(
        image.reshape([-1]).cuda(),
        label.cuda(),
        fc_model,
        epsilon,
        T,
        lr,
        lr_sigma,
        c=c,
        lr_c=lr_c,
        perturb_freq=perturb_freq,
        noise_scale=noise_scale,
        grad_threshold=grad_threshold,
        c_reduce_rate=c_reduce_rate,
        set_proper_sigma_freq=16,
    )
    
    adagd_logits = model(attacked_image_ncvx.reshape([1, 28, 28]))
    adagd_label = torch.argmax(adagd_logits[0])
    adagd_loss = entropy_loss(adagd_logits[0], label)
    
    success = int(adagd_label != label)
    success_count += success
    
    attack_results.append({
        "index": i + 1,
        "clean_label": clean_label.item(),
        "true_label": label.item(),
        "adagd_label": adagd_label.item(),
        "clean_loss": clean_loss.item(),
        "adagd_loss": adagd_loss.item(),
        "success": success
    })
    
    print(f"Attack {i + 1}:")
    print(f"\tClean classification label: {clean_label}")
    print(f"\tGround truth label: {label}")
    print(f"\tUODR-GD classification label: {adagd_label}")
    print(f"\tClean objective: {clean_loss:.3f}")
    print(f"\tUODR-GD objective: {adagd_loss:.3f}")
    print(f"\tAttack Success: {'Yes' if success else 'No'}\n")

success_rate = (success_count / 10) * 100
print(f"Total Attack Success Rate: {success_rate:.2f}%")


Classifier loaded!
Evaluating...
Attack 1:
	Clean classification label: 3
	Ground truth label: 3
	UODR-GD classification label: 7
	Clean objective: 0.000
	UODR-GD objective: 6.850
	Attack Success: Yes

Attack 2:
	Clean classification label: 2
	Ground truth label: 2
	UODR-GD classification label: 6
	Clean objective: 0.000
	UODR-GD objective: 8.931
	Attack Success: Yes

Attack 3:
	Clean classification label: 7
	Ground truth label: 7
	UODR-GD classification label: 0
	Clean objective: 0.000
	UODR-GD objective: 9.017
	Attack Success: Yes

Attack 4:
	Clean classification label: 3
	Ground truth label: 3
	UODR-GD classification label: 9
	Clean objective: 0.000
	UODR-GD objective: 1.046
	Attack Success: Yes

Attack 5:
	Clean classification label: 4
	Ground truth label: 4
	UODR-GD classification label: 9
	Clean objective: 0.000
	UODR-GD objective: 0.958
	Attack Success: Yes

Attack 6:
	Clean classification label: 1
	Ground truth label: 1
	UODR-GD classification label: 2
	Clean objective: 0.000
	

In [67]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.optim import Adam
from torch.utils.data import DataLoader

class CNN_with_Softplus(nn.Module):
    def __init__(self, input_size, input_dim,
                 hiddens_config,
                 fc_hiddens_config = [100, 10]):
        super(CNN_with_Softplus, self).__init__()
        self.input_size = input_size
        self.input_dim = input_dim
        self.hiddens_config = hiddens_config
        self.conv_latent_dim = input_dim
        self.conv_latent_size = input_size
        
        self.ConvLayers = []
        for hidden_params in self.hiddens_config:
            in_dim, out_dim, kernel_dim = hidden_params
            self.ConvLayers.append(nn.Conv2d(in_dim, out_dim, kernel_dim).cuda())
            self.ConvLayers.append(nn.Softplus().cuda())
            self.conv_latent_dim = out_dim
            self.conv_latent_size -= (kernel_dim - 1)
            assert self.conv_latent_size > 0
#             print (self.conv_latent_dim * (self.conv_latent_size ** 2))
        self.ConvLayers = nn.Sequential(*self.ConvLayers)
            
        self.conv_final_dim = self.conv_latent_dim * (self.conv_latent_size ** 2)
        self.fc_layer = []
        current_dim = self.conv_final_dim
        for h in fc_hiddens_config:
            self.fc_layer.append(nn.Linear(current_dim, h).cuda())
            self.fc_layer.append(nn.Softplus().cuda())
            current_dim = h
        self.fc_layer = nn.Sequential(*(self.fc_layer[:-1]))       
        
    def forward(self,x):
        out = self.ConvLayers(x)
        out = out.view(-1, self.conv_final_dim)
        out = self.fc_layer(out)

        return out


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

model_softplus = CNN_with_Softplus(input_size, input_dim, hiddens_config)
model_softplus = model_softplus.cuda()

entropy_loss = nn.CrossEntropyLoss()
optimizer = Adam(model_softplus.parameters(), lr=0.001)


train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

num_epochs = 5
for epoch in range(num_epochs):
    model_softplus.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.cuda(), labels.cuda()

        optimizer.zero_grad()
        outputs = model_softplus(images)
        loss = entropy_loss(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training finished!")

model_softplus.eval()
num_correct, num_all = 0, 0
for images, labels in test_loader:
    images, labels = images.cuda(), labels.cuda()
    outputs = model_softplus(images)
    _, predicted = torch.max(outputs, 1)
    num_all += labels.size(0)
    num_correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {num_correct / num_all * 100:.2f}%")

Epoch [1/5], Loss: 0.3792
Epoch [2/5], Loss: 0.0949
Epoch [3/5], Loss: 0.0678
Epoch [4/5], Loss: 0.0511
Epoch [5/5], Loss: 0.0445
Training finished!
Test Accuracy: 98.31%


In [61]:
image, label = draw_image_randomly(test_loader)
clean_logits = model_softplus(image.reshape([1, *image.shape]))
clean_loss = entropy_loss(clean_logits,
                          label.reshape([1, *label.shape]))

print ("Randomly draw an image...")
print ("\tClean classification label: {}\n\tGround truth label: {}, objective: {:.3f}\n".format(torch.argmax(clean_logits[0]), 
                                                                                                label, clean_loss))

attacked_image = pgd_atk(image.reshape([1, *image.shape]), 
                         label.reshape([1, *label.shape]))
pgd_loss = entropy_loss( model_softplus(attacked_image)[0], label)
pgd_label = torch.argmax(model_softplus(attacked_image)[0])
print ("Epsilon: {}\n".format(epsilon))
print ("PGD attacking...")
print ("\tClassification label: {}\n\tobjective: {:.3f}\n".format(pgd_label, pgd_loss))

Randomly draw an image...
	Clean classification label: 9
	Ground truth label: 9, objective: 0.002

Epsilon: 0.25

PGD attacking...
	Classification label: 3
	objective: 1.933



In [62]:
epsilon = 0.2
T = 500
lr = 10e-1 
lr_c = 1e-4
lr_sigma = 2.5e-1 
c = 0.5
perturb_freq = 5
noise_scale = 1e-3
grad_threshold = 2
c_reduce_rate = 1e-3 
attacked_image_ncvx, obj_cvx = adversarial_attack_nonconvexOpt(image.reshape([-1]).cuda(), 
                                                 label.cuda(),convert_CNN_to_FC(model_softplus, [28, 28, 1]), 
                                                 epsilon, T, lr, lr_sigma,
                                                 c = c, lr_c = lr_c,
                                                 perturb_freq = perturb_freq,
                                                 noise_scale = noise_scale,
                                                 grad_threshold = grad_threshold,
                                                 c_reduce_rate = c_reduce_rate, set_proper_sigma_freq = 16)
print ("ADR-GD attacking...")
adagd_label = torch.argmax(model_softplus(attacked_image_ncvx.reshape([1, 28, 28])))
print ("\tClassification label: {}\n\tobjective: {:.3f}\n".format(adagd_label, obj_cvx))

ADR-GD attacking...
	Classification label: 9
	objective: 1.711



In [63]:
from models import *
from helpers1 import * # UODR-GD
from torchattacks import PGD, FGSM

import os, sys

current_dir = os.getcwd()
path_to_append = os.path.join(current_dir, "configs")
if path_to_append not in sys.path:
    sys.path.append(path_to_append)

from importlib import import_module
def load_configs_from_file(config_file):
    configs = import_module(config_file.split(".")[0])
    
    return configs.model_path, configs.hiddens_config, configs.batch_size, configs.epsilon, configs.T, configs.c, configs.lr, configs.lr_sigma, \
        configs.lr_c, configs.perturb_freq, configs.noise_scale, configs.grad_threshold, configs.c_reduce_rate

"""Can replace with `adv_MNIST_clean_config` or `adv_MNIST_pgd_config`"""
# config_filename = "adv_MNIST_clean_config" 
config_filename = "adv_MNIST_pgd_config" 

input_dim, input_size = 1, 28 # MNIST images
model_path, hiddens_config, batch_size, epsilon, T, c, lr, lr_sigma, lr_c, perturb_freq, noise_scale, grad_threshold, c_reduce_rate  = load_configs_from_file(config_filename)

epsilon = 0.25
T = 1000
lr = 10e-1 
lr_c = 1e-4
lr_sigma = 2.5e-1 
c = 0.5
perturb_freq = 5
noise_scale = 1e-3
c_reduce_rate = 1e-3 
batch_size = 64

entropy_loss = nn.CrossEntropyLoss()

attacked_image_ncvx0, obj_cvx = adversarial_attack_nonconvexOpt(image.reshape([-1]).cuda(), 
                                                 label.cuda(), convert_CNN_to_FC(model_softplus, [28, 28, 1]),
                                                 epsilon, T, lr, lr_sigma,
                                                 c = c, lr_c = lr_c,
                                                 perturb_freq = perturb_freq,
                                                 noise_scale = noise_scale,
                                                 grad_threshold = grad_threshold,
                                                 c_reduce_rate = c_reduce_rate, set_proper_sigma_freq = 16)
print ("UODR-GD attacking...")
adagd_label0 = torch.argmax(model_softplus(attacked_image_ncvx0.reshape([1, 28, 28])))
print ("\tClassification label: {}\n\tobjective: {:.3f}\n".format(adagd_label0, obj_cvx))

UODR-GD attacking...
	Classification label: 7
	objective: 3.318

